# NCKH 03 — Results, Confusion Matrices and Error Analysis

The selected configuration is evaluated on a strictly later test
period. This notebook focuses on the evidence needed to compare the
proposed LightGBM model with the baselines and to inspect high-risk
errors.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if not (ROOT / "reports" / "nckh_online_retail_multiclass").exists():
    ROOT = Path("..").resolve()
REPORT = ROOT / "reports" / "nckh_online_retail_multiclass"
SELECTED = REPORT / "selected"
FIG_DIR = REPORT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CRITICAL", "HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]
CLASS_COLORS = {
    "CRITICAL": "#B42318",
    "HIGH_RISK": "#E76F00",
    "MEDIUM_RISK": "#176B87",
    "LOW_RISK": "#2A9D8F",
}
MODEL_LABELS = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "lightgbm": "LightGBM",
}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

with open(SELECTED / "dataset_profile.json", encoding="utf-8") as handle:
    profile = json.load(handle)
with open(REPORT / "selected_config.json", encoding="utf-8") as handle:
    selected_config = json.load(handle)
with open(SELECTED / "metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

In [ ]:
predictions = pd.read_csv(SELECTED / "test_predictions.csv")
lightgbm_metrics = metrics["models"]["lightgbm"]["test"]
predictions["actual_class"] = predictions["risk_class"]
predictions["high_risk_score"] = (
    predictions["lightgbm_prob_CRITICAL"] + predictions["lightgbm_prob_HIGH_RISK"]
)
print(predictions.shape)
print(predictions["cutoff_date"].min(), "to", predictions["cutoff_date"].max())

## Figure 8 — Per-class precision, recall and F1

In [ ]:
per_class = pd.DataFrame(lightgbm_metrics["per_class"]).T[["precision", "recall", "f1"]]
fig, ax = plt.subplots(figsize=(9.5, 5.0))
x = np.arange(len(CLASS_NAMES))
width = 0.24
for index, metric_name in enumerate(["precision", "recall", "f1"]):
    bars = ax.bar(x + (index - 1) * width, per_class.loc[CLASS_NAMES, metric_name],
                  width, label=metric_name.title(),
                  color=["#264653", "#E76F00", "#2A9D8F"][index])
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
ax.set_xticks(x, CLASS_NAMES)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("LightGBM test performance by risk class")
ax.legend(frameon=False, ncol=3)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "08_lightgbm_per_class_metrics.png", bbox_inches="tight")
plt.show()

## Figure 9 — Confusion matrices

In [ ]:
matrix_models = ["recent_quantile_rule", "logistic_regression", "random_forest", "lightgbm"]
fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.8))
for ax, name in zip(axes.flat, matrix_models):
    matrix = np.asarray(metrics["models"][name]["test"]["confusion_matrix"])
    image = ax.imshow(matrix, cmap="YlOrBr")
    ax.set_title(MODEL_LABELS[name])
    ax.set_xticks(range(4), CLASS_NAMES, rotation=35, ha="right")
    ax.set_yticks(range(4), CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    threshold = matrix.max() * 0.55
    for row in range(4):
        for col in range(4):
            ax.text(col, row, f"{matrix[row, col]:,}",
                    ha="center", va="center",
                    color="white" if matrix[row, col] > threshold else "#212529",
                    fontsize=8)
fig.suptitle("Test confusion matrices on the selected horizon", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_confusion_matrices.png", bbox_inches="tight")
plt.show()

## Figure 10 — Risk score distribution and error concentration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for class_name in CLASS_NAMES:
    values = predictions.loc[predictions["actual_class"] == class_name, "high_risk_score"]
    axes[0].hist(values, bins=25, alpha=0.55, density=True,
                 color=CLASS_COLORS[class_name], label=class_name)
axes[0].set_xlabel("P(CRITICAL) + P(HIGH_RISK)")
axes[0].set_ylabel("Density")
axes[0].set_title("LightGBM high-risk score by actual class")
axes[0].legend(frameon=False, fontsize=8)
axes[0].grid(axis="y", alpha=0.2)

actual_high = predictions["actual_class"].isin(["CRITICAL", "HIGH_RISK"])
predicted_high = predictions["lightgbm_prediction_name"].isin(["CRITICAL", "HIGH_RISK"])
error_frame = pd.crosstab(
    predictions.loc[actual_high, "actual_class"],
    predictions.loc[actual_high & ~predicted_high, "lightgbm_prediction_name"],
).reindex(index=["CRITICAL", "HIGH_RISK"], fill_value=0)
if error_frame.shape[1]:
    error_frame.plot(kind="bar", stacked=True, ax=axes[1],
                     color=[CLASS_COLORS.get(label, "#6C757D") for label in error_frame.columns])
axes[1].set_title("Missed high-risk cases by predicted class")
axes[1].set_xlabel("Actual high-risk class")
axes[1].set_ylabel("Number of cases")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Predicted", frameon=False, fontsize=7)
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "10_risk_scores_and_errors.png", bbox_inches="tight")
plt.show()

**Reading the errors.** A `CRITICAL` or `HIGH_RISK` row predicted as a
lower class is a missed high-risk case; a lower-risk row predicted as
high risk is a review workload. These are classification errors under
the sales-based label, not evidence of inventory shortage or excess.